In [ ]:
# ============================================================
# WEEKLY + MONTHLY SCANNER
# S&P 500 + ETFs + COMMODITIES + FOREX
#
# CONDITIONS:
# 1. Latest completed weekly candle is green.
# 2. Weekly close > weekly SMA(20).
# 3. Weekly MACD(12,26) > signal(9).
# 4. Weekly MACD > 0.
# 5. At least one of last 4 completed weekly closes
#    was below its own weekly SMA(20).
# 6. Weekly close > completed-month SMA(20).
# ============================================================
%pip -q install yfinance pandas lxml requests

import io
import time
import requests
import pandas as pd
import yfinance as yf
from IPython.display import display

# ---------------- SETTINGS ----------------
SMA_LENGTH = 20
MONTHLY_SMA_LENGTH = 20
MACD_FAST = 12
MACD_SLOW = 26
MACD_SIGNAL = 9
LOOKBACK_CANDLES = 4
HISTORY_YEARS = 5
MINIMUM_WEEKS = 60
BATCH_SIZE = 30

# A week becomes eligible Saturday at 00:00 New York time.
# This excludes unfinished Friday forex/futures sessions.
now = pd.Timestamp.now(tz="America/New_York")
today = now.tz_localize(None).normalize()

last_friday = today - pd.Timedelta(
    days=(today.weekday() - 4) % 7
)
if today.weekday() == 4:
    last_friday -= pd.Timedelta(days=7)

LAST_COMPLETED_WEEK = last_friday

# Latest calendar month completed by the weekly candle date.
month_of_week = LAST_COMPLETED_WEEK.to_period("M")
if LAST_COMPLETED_WEEK == month_of_week.end_time.normalize():
    LAST_COMPLETED_MONTH = month_of_week
else:
    LAST_COMPLETED_MONTH = month_of_week - 1

START_DATE = (
    now - pd.DateOffset(years=HISTORY_YEARS)
).date()

# ---------------- BUILD UNIVERSE ----------------
universe = {}

def add_asset(yahoo, name, category, tradingview=None):
    universe[yahoo] = {
        "Name": name,
        "Category": category,
        "TradingView": tradingview or yahoo,
    }

# Current S&P 500 constituents
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
)
response.raise_for_status()

tables = pd.read_html(io.StringIO(response.text))
sp500 = next(
    table for table in tables
    if {"Symbol", "Security"}.issubset(table.columns)
)

for _, row in sp500.iterrows():
    symbol = str(row["Symbol"])
    add_asset(
        symbol.replace(".", "-"),
        row["Security"],
        "S&P 500",
        symbol,
    )

# Broad ETF selection, not every listed ETF.
etfs = """
SPY VOO IVV RSP SPLG QQQ QQQM DIA IWM IWB IWR IJR IJH
VTI ITOT SCHB VT VUG VTV VIG VYM SCHD DGRO HDV DVY
DGRW SDY NOBL SPYD JEPI JEPQ DIVO SPYI QQQI
XLK XLF XLE XLV XLI XLB XLP XLY XLU XLRE XLC
VGT VFH VHT VDE VIS VAW VDC VCR VPU VOX
SMH SOXX SOXQ IGV XSD XBI IBB IHI XRT ITB XHB
KRE KBE XOP OIH IYT JETS TAN ICLN LIT URA URNM
PAVE CIBR HACK SKYY CLOU BOTZ ROBO AIQ ARKK ARKW ARKG
MTUM QUAL USMV VLUE SIZE SPHQ IWF IWD IWO IWN
EFA EFG EFV IEFA EEM IEMG VWO VXUS VEA
EWJ EWZ EWW EWC EWA EWG EWU EWT EWY INDA MCHI FXI KWEB
VNQ VNQI IYR SCHH REM REET
BND AGG TLT IEF SHY GOVT VGIT VGLT
TIP SCHP VTIP LQD HYG JNK BKLN EMB MUB
SGOV BIL USFR TFLO SHV
GLD IAU GLDM SLV SIVR GDX GDXJ SIL SILJ
DBC PDBC GSG COMT DBA DBB CPER USO BNO UNG
CORN WEAT SOYB
IBIT FBTC ARKB BITB ETHA FETH
UUP UDN FXE FXY FXB FXA FXC FXF
""".split()

for symbol in dict.fromkeys(etfs):
    add_asset(symbol, symbol, "ETF")

# Continuous commodity futures.
commodities = {
    "GC=F": ("Gold", "COMEX:GC1!"),
    "SI=F": ("Silver", "COMEX:SI1!"),
    "HG=F": ("Copper", "COMEX:HG1!"),
    "PL=F": ("Platinum", "NYMEX:PL1!"),
    "PA=F": ("Palladium", "NYMEX:PA1!"),
    "CL=F": ("WTI crude oil", "NYMEX:CL1!"),
    "BZ=F": ("Brent crude oil", "NYMEX:BZ1!"),
    "NG=F": ("Natural gas", "NYMEX:NG1!"),
    "RB=F": ("Gasoline", "NYMEX:RB1!"),
    "HO=F": ("Heating oil", "NYMEX:HO1!"),
    "ZC=F": ("Corn", "CBOT:ZC1!"),
    "ZW=F": ("Wheat", "CBOT:ZW1!"),
    "ZS=F": ("Soybeans", "CBOT:ZS1!"),
    "ZM=F": ("Soybean meal", "CBOT:ZM1!"),
    "ZL=F": ("Soybean oil", "CBOT:ZL1!"),
    "KC=F": ("Coffee", "ICEUS:KC1!"),
    "CC=F": ("Cocoa", "ICEUS:CC1!"),
    "SB=F": ("Sugar", "ICEUS:SB1!"),
    "CT=F": ("Cotton", "ICEUS:CT1!"),
    "LE=F": ("Live cattle", "CME:LE1!"),
    "GF=F": ("Feeder cattle", "CME:GF1!"),
    "HE=F": ("Lean hogs", "CME:HE1!"),
}

for yahoo, (name, tv) in commodities.items():
    add_asset(yahoo, name, "Commodity", tv)

# Forex majors and crosses.
forex_pairs = """
EURUSD GBPUSD AUDUSD NZDUSD USDJPY USDCHF USDCAD
EURGBP EURJPY EURCHF EURAUD EURNZD EURCAD
GBPJPY GBPCHF GBPAUD GBPNZD GBPCAD
AUDJPY AUDCHF AUDCAD AUDNZD
NZDJPY NZDCHF NZDCAD
CADJPY CADCHF CHFJPY
""".split()

for pair in forex_pairs:
    add_asset(
        pair + "=X",
        pair[:3] + "/" + pair[3:],
        "Forex",
        "OANDA:" + pair,
    )

tickers = list(universe)

print(f"Completed week ending: {LAST_COMPLETED_WEEK.date()}")
print(f"Monthly SMA through:   {LAST_COMPLETED_MONTH}")
print(f"Total symbols:         {len(tickers)}")
print(
    pd.Series([
        item["Category"] for item in universe.values()
    ]).value_counts().to_string()
)
print("\nDownloading prices...\n")

# ---------------- DOWNLOAD ----------------
prices = {}
errors = {}

def extract_bars(data, ticker):
    if data is None or data.empty:
        return pd.DataFrame()

    if isinstance(data.columns, pd.MultiIndex):
        if ticker not in data.columns.get_level_values(0):
            return pd.DataFrame()
        bars = data[ticker][["Open", "Close"]].copy()
    else:
        bars = data[["Open", "Close"]].copy()

    return bars.dropna().sort_index()

def download(symbols):
    return yf.download(
        tickers=symbols,
        start=str(START_DATE),
        interval="1d",
        auto_adjust=True,
        group_by="ticker",
        multi_level_index=True,
        threads=4,
        progress=False,
        timeout=30,
    )

for offset in range(0, len(tickers), BATCH_SIZE):
    batch = tickers[offset:offset + BATCH_SIZE]

    try:
        data = download(batch)
        for ticker in batch:
            bars = extract_bars(data, ticker)
            if not bars.empty:
                prices[ticker] = bars
    except Exception as exc:
        for ticker in batch:
            errors[ticker] = str(exc)

    print(
        f"Requested "
        f"{min(offset + BATCH_SIZE, len(tickers))}/{len(tickers)}"
    )
    time.sleep(1)

missing = [ticker for ticker in tickers if ticker not in prices]
print(f"\nRetrying {len(missing)} missing symbols...")

for ticker in missing:
    for attempt in range(2):
        try:
            bars = extract_bars(download([ticker]), ticker)
            if bars.empty:
                raise ValueError("No price data returned")

            prices[ticker] = bars
            errors.pop(ticker, None)
            break
        except Exception as exc:
            errors[ticker] = str(exc)
            time.sleep(attempt + 1)

# ---------------- SCAN ----------------
matches = []
skipped = []
diagnostics = []

for ticker in tickers:
    info = universe[ticker]

    try:
        if ticker not in prices:
            raise ValueError(errors.get(ticker, "Download failed"))

        daily = prices[ticker].copy()

        # Preserve provider daily session dates.
        index = pd.DatetimeIndex(daily.index)
        if index.tz is not None:
            index = index.tz_localize(None)

        daily.index = index.normalize()
        daily = daily[~daily.index.duplicated(keep="last")]

        # Exclude all data after the evaluated weekly candle.
        daily = daily.loc[:LAST_COMPLETED_WEEK]

        weekly = daily.resample("W-FRI").agg({
            "Open": "first",
            "Close": "last",
        }).dropna()

        if len(weekly) < MINIMUM_WEEKS:
            raise ValueError(
                f"Insufficient history: {len(weekly)} weekly candles"
            )

        if weekly.index[-1] != LAST_COMPLETED_WEEK:
            raise ValueError("Missing latest completed weekly candle")

        # Thursday is accepted for Friday market holidays.
        final_week_start = LAST_COMPLETED_WEEK - pd.Timedelta(days=4)
        final_week_daily = daily.loc[
            final_week_start:LAST_COMPLETED_WEEK
        ]

        if (
            final_week_daily.empty
            or final_week_daily.index[-1]
            < LAST_COMPLETED_WEEK - pd.Timedelta(days=1)
        ):
            raise ValueError("Latest weekly data appears stale")

        # Monthly closes, excluding incomplete months.
        # Period grouping avoids month-end resampling aliases.
        monthly = daily["Close"].groupby(
            daily.index.to_period("M")
        ).last()

        monthly = monthly.loc[:LAST_COMPLETED_MONTH]

        if len(monthly) < MONTHLY_SMA_LENGTH:
            raise ValueError(
                f"Insufficient monthly history: {len(monthly)} months"
            )

        expected_months = pd.period_range(
            end=LAST_COMPLETED_MONTH,
            periods=MONTHLY_SMA_LENGTH,
            freq="M",
        )
        monthly_window = monthly.reindex(expected_months)

        if monthly_window.isna().any():
            raise ValueError("Missing month in monthly SMA window")

        monthly_sma = monthly_window.mean()

        # Weekly indicators.
        close = weekly["Close"]
        sma = close.rolling(SMA_LENGTH).mean()

        macd = (
            close.ewm(span=MACD_FAST, adjust=False).mean()
            - close.ewm(span=MACD_SLOW, adjust=False).mean()
        )
        signal = macd.ewm(
            span=MACD_SIGNAL, adjust=False
        ).mean()

        recent_close = close.tail(LOOKBACK_CANDLES)
        recent_sma = sma.reindex(recent_close.index)
        below = recent_close < recent_sma
        below_dates = recent_close.index[below]

        green = bool(
            close.iloc[-1] > weekly["Open"].iloc[-1]
        )
        above_weekly_sma = bool(close.iloc[-1] > sma.iloc[-1])
        above_monthly_sma = bool(close.iloc[-1] > monthly_sma)
        above_signal = bool(macd.iloc[-1] > signal.iloc[-1])
        above_zero = bool(macd.iloc[-1] > 0)
        recent_pullback = bool(below.any())

        qualifies = all([
            green,
            above_weekly_sma,
            above_monthly_sma,
            above_signal,
            above_zero,
            recent_pullback,
        ])

        diagnostics.append({
            "Symbol": ticker,
            "Category": info["Category"],
            "Green weekly candle": green,
            "Above weekly SMA20": above_weekly_sma,
            "Above monthly SMA20": above_monthly_sma,
            "MACD above signal": above_signal,
            "MACD above zero": above_zero,
            "Recent close below weekly SMA20": recent_pullback,
            "Matches": qualifies,
        })

        if qualifies:
            matches.append({
                "Symbol": ticker,
                "TradingView": info["TradingView"],
                "Name": info["Name"],
                "Category": info["Category"],
                "Week ending": LAST_COMPLETED_WEEK.strftime("%Y-%m-%d"),
                "Weekly open": weekly["Open"].iloc[-1],
                "Weekly close": close.iloc[-1],
                "Weekly SMA20": sma.iloc[-1],
                "Monthly SMA20": monthly_sma,
                "Monthly SMA through": str(LAST_COMPLETED_MONTH),
                "% above weekly SMA20": (
                    close.iloc[-1] / sma.iloc[-1] - 1
                ) * 100,
                "% above monthly SMA20": (
                    close.iloc[-1] / monthly_sma - 1
                ) * 100,
                "MACD": macd.iloc[-1],
                "Signal": signal.iloc[-1],
                "Histogram": macd.iloc[-1] - signal.iloc[-1],
                "Below-SMA candles": int(below.sum()),
                "Last below-SMA week": below_dates[-1].strftime(
                    "%Y-%m-%d"
                ),
            })

    except Exception as exc:
        skipped.append({
            "Symbol": ticker,
            "Category": info["Category"],
            "Reason": str(exc),
        })

# ---------------- RESULTS ----------------
columns = [
    "Symbol", "TradingView", "Name", "Category", "Week ending",
    "Weekly open", "Weekly close", "Weekly SMA20", "Monthly SMA20",
    "Monthly SMA through", "% above weekly SMA20",
    "% above monthly SMA20", "MACD", "Signal", "Histogram",
    "Below-SMA candles", "Last below-SMA week",
]

results = pd.DataFrame(matches, columns=columns)
diagnostics_df = pd.DataFrame(diagnostics)

if not results.empty:
    results = results.sort_values(
        ["Category", "% above weekly SMA20", "Symbol"]
    ).reset_index(drop=True)

print("\n" + "=" * 70)
print(f"Successfully scanned: {len(diagnostics)}/{len(tickers)}")
print(f"Matching symbols:     {len(results)}")
print(f"Skipped symbols:      {len(skipped)}")
print("=" * 70)

watchlist = ",".join(results["TradingView"])

if results.empty:
    print("\nNo symbols meet all conditions.")
else:
    display(results.round(5))

    print("\nTRADINGVIEW WATCHLIST — ALL MATCHES:\n")
    print(watchlist)

    for category, group in results.groupby("Category"):
        print(f"\n{category.upper()}:\n")
        print(",".join(group["TradingView"]))

if not diagnostics_df.empty:
    print("\nNUMBER PASSING EACH INDIVIDUAL CONDITION:")
    filter_columns = [
        "Green weekly candle",
        "Above weekly SMA20",
        "Above monthly SMA20",
        "MACD above signal",
        "MACD above zero",
        "Recent close below weekly SMA20",
        "Matches",
    ]
    display(
        diagnostics_df.groupby("Category")[filter_columns]
        .sum()
        .astype(int)
    )

# Save results.
results.to_csv(
    "/content/weekly_monthly_scanner_matches.csv",
    index=False,
)
diagnostics_df.to_csv(
    "/content/weekly_monthly_scanner_diagnostics.csv",
    index=False,
)

with open("/content/tradingview_watchlist.txt", "w") as file:
    file.write(watchlist)

if skipped:
    skipped_df = pd.DataFrame(skipped)
    print("\nSKIPPED SYMBOLS:")
    display(skipped_df)
    skipped_df.to_csv(
        "/content/weekly_monthly_scanner_skipped.csv",
        index=False,
    )

print("\nSaved CSV results and tradingview_watchlist.txt")

# Uncomment to download the watchlist automatically:
# from google.colab import files
# files.download("/content/tradingview_watchlist.txt")